# 2026F - San Jose State University - CMPE-257 - project - Chris Carl

## Metadata
<div style="display: inline-block">
<!-- get the table rendered to the left rather than default center, the spaces after the raw html are important for markdown https://stackoverflow.com/a/78525025 -->

| Key | Value |
| :--- | :--- |
| author | Chris Carl(mailto:chris.carl@sjsu.edu) |
| date | 2026-11-13 |
| dataset | [AmesHousing.txt, save as CSV](https://jse.amstat.org/v19n3/decock/AmesHousing.txt) © 2015 American Statistical Association |

<!-- get the table rendered to the left rather than default center, the spaces after the raw html are important for markdown https://stackoverflow.com/a/78525025 -->
</div>

<!--
commands:
    ipynb CMPE-257/assignments/project/project.ipynb  # chriscarl.tools.documents
-->


# Abstract

This is my submission of project of CMPE-257 at SJSU instructed by Dr. Zara Hajihashemi in Fall 2026.

Explain the executive summary and findings. You HAVE TO. this is the first thing they see.


# TOC


<!-- # SUGGESTED TOC ORGANIZATION

# Project
# Abstract
# Table of Contents
# Question
# Answer
# Solution
## Background
## The Plan
## Code - Environment
## Code - Setup
## Code - Functions
## Code - Main
# Post Mortem
## Caveats
## Improvements
## Replication
# Appendix
## Criterion
## Changelog
-->

# Solution


## Code - Setup

The following is boilerplate I've accumulated over the years and can be safely disregarded by the reader until it becomes relevant as part of deeper inquiry.


In [1]:
# stdlib imports
import __main__
import os
import sys
import pprint
import importlib
import subprocess
# hyper-critical imports
try:
    from IPython import get_ipython
    IPYTHON_SHELL = get_ipython()  # None if running in vanilla python
except ImportError:
    IPYTHON_SHELL = None

SCRIPT_FILEPATH = ''
SCRIPT_FILENAME = ''
if IPYTHON_SHELL:
    SCRIPT_DIRPATH = str(IPYTHON_SHELL.run_line_magic('pwd', ' # %pwd is a "magic" command  https://ipython.readthedocs.io/en/stable/interactive/magics.html'))
    if hasattr(__main__, '__vsc_ipynb_file__'):  # vscode
        SCRIPT_FILEPATH = __main__.__vsc_ipynb_file__
    elif hasattr(__main__, '__session__'):  # localhost:8888/notebooks
        SCRIPT_FILEPATH = __main__.__session__
else:
    SCRIPT_FILEPATH = __file__
    SCRIPT_DIRPATH = os.path.dirname(SCRIPT_FILEPATH)

SCRIPT_FILENAME = os.path.splitext(os.path.basename(SCRIPT_FILEPATH))[0]

pprint.pprint(dict(dir=SCRIPT_DIRPATH, fp=SCRIPT_FILEPATH, fn=SCRIPT_FILENAME), indent=2)

AUTO_INSTALL = False
packages = {}  # {'Werkzeug': 'werkzeug'}  # package name: module name
modules = ['matplotlib', 'pandas', 'ipykernel', 'ipympl', 'ipywidgets']
packages.update({m: m for m in modules})
package, module = '', ''
install_string = ' '.join(packages)
try:
    for package, module in packages.items():
        importlib.import_module(module)
except ImportError:
    print(f'"pip install {package}"', file=sys.stderr)
    if AUTO_INSTALL:
        %pip install {install_string}
    else:
        raise

# third party imports
import matplotlib.pyplot as plt

EXES = {
    'pandoc': {'exists': False},
    'latex': {'exists': False, 'win32': 'miktex', 'linux': 'texlive'},
}

for exe, data in EXES.items():
    if IPYTHON_SHELL:
        if sys.platform == 'win32':
            _res = !where.exe {exe}
            _exists = not _res[0].startswith('INFO')
        else:
            _res = !which {exe}
            _exists = bool(_res)
    else:
        if sys.platform == 'win32':
            _res = subprocess.check_output(f'where.exe {exe}', universal_newlines=True)
            _exists = not _res.startswith('INFO')
        else:
            _res = subprocess.check_output(f'which {exe}', universal_newlines=True)
            _exists = bool(_res)
    EXES[exe]['exists'] = _exists

print('pandoc:', EXES['pandoc']['exists'])
if not EXES['pandoc']['exists']:
    print('you must install pandoc!', file=sys.stderr)
    if sys.platform == 'win32':
        print('    choco install pandoc -y', file=sys.stderr)
    else:
        print('    sudo apt install pandoc -y', file=sys.stderr)
        print('    sudo yum install pandoc -y', file=sys.stderr)
        print('    dnf install pandoc -y', file=sys.stderr)
    raise RuntimeError()

print('latex:', EXES['latex']['exists'])
plt.rcParams.update(
    {
        'text.usetex': EXES['latex']['exists'],  # , 'font.family': 'Helvetica'  # requires latex installed
        'text.latex.preamble': r'\usepackage{amsmath}',  # https://stackoverflow.com/a/23856968  \text command
    }

)

# command to tell the notebook to plt.show() IN THE NOTEBOOK, otherwise you call plt.show()
%matplotlib inline
# command to tell the notebook to render interactable matplotlib, requires pip install ipympl
# %matplotlib widget

print('# pip install', install_string)
if not IPYTHON_SHELL:
    raise RuntimeError('Not sure how, but somehow this code is being run through a non-iPython environment!')


{ 'dir': 'c:\\Users\\chris\\src\\sjsu.CMPE-257.ml-social-bot-detection\\src',
  'fn': 'project',
  'fp': 'c:\\Users\\chris\\src\\sjsu.CMPE-257.ml-social-bot-detection\\src\\project.ipynb'}
pandoc: True
latex: True
# pip install matplotlib pandas ipykernel ipympl ipywidgets


# Setup

In [2]:
PROJECT_DIRECTORY = os.path.abspath(os.path.join(SCRIPT_DIRPATH, '..'))
DATASET_DIRECTORIES = [
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\cresci-17'),
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\twibot-22'),
]


In [3]:
import os
import zipfile

def unzip_file(zip_path, extract_to, password=None, print_progress=True):
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP file not found: {zip_path}")
    os.makedirs(extract_to, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            pwd = password.encode() if password else None
            for member in zip_ref.infolist():
                # Prevent Zip Slip vulnerability
                extracted_path = os.path.join(extract_to, member.filename)
                if member.filename.startswith('__MACOSX/._') or member.filename.split('/')[-1].startswith('._'):
                    continue

                if not os.path.realpath(extracted_path).startswith(os.path.realpath(extract_to)):
                    raise Exception(f"Unsafe file path detected: {member.filename}")

                if print_progress:
                    print(member)
                if not os.path.exists(extracted_path):
                    zip_ref.extract(member, path=extract_to, pwd=pwd)

                basename, ext = os.path.splitext(member.filename)

                if ext == '.zip':
                    unzip_file(os.path.join(extract_to, member.filename), os.path.join(extract_to, basename), password=password, print_progress=print_progress)
    except zipfile.BadZipFile:
        raise zipfile.BadZipFile(f"Invalid ZIP file: {zip_path}")
    except RuntimeError as e:
        if "password required" in str(e).lower():
            raise RuntimeError("This ZIP file is password-protected. Please provide a valid password.")
        else:
            raise


In [4]:
ZIPS = [
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\cresci-17\datasets_full.csv.zip'),
]
for file in ZIPS:
    basename, _ = os.path.splitext(file)
    unzip_file(file, basename)


<ZipInfo filename='datasets_full.csv/' filemode='drwxr-xr-x' external_attr=0x10>
<ZipInfo filename='datasets_full.csv/crowdflower_results.csv.zip' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=290528 compress_size=289584>
<ZipInfo filename='crowdflower_results.csv/' filemode='drwxr-xr-x' external_attr=0x4000>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_aggregated.csv' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=617235 compress_size=107077>
<ZipInfo filename='__MACOSX/' filemode='drwxrwxr-x' external_attr=0x4000>
<ZipInfo filename='__MACOSX/crowdflower_results.csv/' filemode='drwxrwxr-x' external_attr=0x4000>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_contributors.csv' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=24118 compress_size=8227>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_detailed.csv' compress_type=deflate filemode='-rw-r--r--' ex

In [5]:
import multiprocessing
import concurrent.futures
from typing import Callable, Any

MAX_WORKERS = multiprocessing.cpu_count() - 1

def parallel(funcs, args, kwargs, key=lambda args: args[0], max_workers=MAX_WORKERS):
    # type: (list[Callable], list[tuple], list[dict], Callable, int) -> list[tuple[Any, Any|Exception]]
    # https://docs.python.org/3/library/concurrent.futures.html
    N = len(funcs)
    if len(funcs) != len(args) or len(args) != len(kwargs):
        raise ValueError('need to provide same list amount of funcs, args, kwargs')

    res = []
    # NOTE: to run this, you need to use external files for the functions you have in mind
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_ret = {executor.submit(funcs[i], *args[i], **kwargs[i]): key(args[i]) for i in range(len(args))}
        for f, future in enumerate(concurrent.futures.as_completed(future_to_ret)):
            arg = future_to_ret[future]
            try:
                data = future.result()
            except Exception as exc:
                print(f'{f+1} / {N} - FAILURE! {arg}: {exc}', file=sys.stderr)
                res += [(arg, exc)]
            else:
                print(f'{f+1} / {N} - SUCCESS! {arg}: {data}', file=sys.stdout)
                res += [(arg, data)]
    return res


In [6]:
%%writefile data_to_pickle.py
import os
import pickle
import warnings
import pandas as pd

def data_to_pickle(filepath, pickle_filepath):
    # type: (str, str) -> list[tuple[str, str]]
    changed_columns = []
    # if os.path.isfile(pickle_filepath) and os.path.getsize(filepath) / os.path.getsize(pickle_filepath) > 0.3:
    #     changed_columns += [('pickle', 'exists')]
    #     return changed_columns

    ext = os.path.splitext(filepath)[1]
    if ext.lower() == '.json':
        try:
            df = pd.read_json(filepath, encoding='utf-8', orient='records')
        except UnicodeDecodeError:
            df = pd.read_json(filepath, encoding='latin-1', orient='records')
    elif ext.lower() == '.csv':
        try:
            df = pd.read_csv(filepath, encoding='utf-8', low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(filepath, encoding='latin-1', low_memory=False)
    else:
        raise NotImplementedError(f'no parser implemented for {ext.lower()!r}')

    for col in df.select_dtypes(include=['object'], exclude=['str']).columns:
        df[col] = df[col].astype('category')
        changed_columns.append((col, 'category'))

    for col in df.columns:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                df[col] = pd.to_datetime(df[col])
                changed_columns.append((col, 'datetime'))
        except Exception:
            pass
        if df[col].nunique() < 100:
            df[col] = df[col].astype('category')
            changed_columns.append((col, 'category'))

    with open(pickle_filepath, 'wb') as f:
        pickle.dump(df, f)

    return changed_columns

Overwriting data_to_pickle.py


In [ ]:
import os
import glob
import pandas as pd
from data_to_pickle import data_to_pickle


funcs, args, kwargs = [], [], []
data_to_pickle_filepaths = {}
for d, dataset_directory in enumerate(DATASET_DIRECTORIES):
    globs = list(glob.glob(os.path.join(dataset_directory, '**', '*.csv'), recursive=True))
    globs += list(glob.glob(os.path.join(dataset_directory, '**', '*.json'), recursive=True))
    globs = sorted(globs, key=lambda pth: os.path.getsize(pth), reverse=True)  # complete in greatest to least
    for c, data_filepath in enumerate(globs):
        if not os.path.isfile(data_filepath):  # could be named datasets_full.csv...
            continue
        filename = os.path.splitext(data_filepath)[0]
        pickle_filepath = f'{filename}.pkl'

        funcs += [data_to_pickle]
        args += [(data_filepath, pickle_filepath)]
        kwargs += [dict()]

        data_to_pickle_filepaths[data_filepath] = pickle_filepath

results = parallel(funcs, args, kwargs, key=lambda args: args[0], max_workers=2)

errors = []
original, pickled = 0, 0
for t, tpl in enumerate(results):
    data_filepath, result = tpl
    pickle_filepath = data_to_pickle_filepaths[data_filepath]
    print(f'{t+1} / {len(results)} - "{os.path.relpath(data_filepath, PROJECT_DIRECTORY)}"')
    if isinstance(result, Exception):
        errors.append(data_filepath)
    else:
        original += os.path.getsize(data_filepath)
        pickled += os.path.getsize(pickle_filepath)

print(f'saved {((original - pickled) / original * 100):0.3f}% from {original / 1024**3:0.3f}GB to {pickled / 1024**3:0.3f}GB')
for error in errors:
    print(f'  - "{error}"')


1 / 34 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\tweets.csv: Error tokenizing data. C error: Expected 26 fields in line 8, saw 27



2 / 34 - SUCCESS! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv: [('id', 'datetime'), ('source', 'category'), ('user_id', 'datetime'), ('truncated', 'datetime'), ('truncated', 'category'), ('in_reply_to_status_id', 'datetime'), ('in_reply_to_user_id', 'datetime'), ('retweeted_status_id', 'datetime'), ('place', 'category'), ('contributors', 'datetime'), ('contributors', 'category'), ('retweet_count', 'datetime'), ('reply_count', 'datetime'), ('reply_count', 'category'), ('favorite_count', 'datetime'), ('favorite_count', 'category'), ('favorited', 'datetime'), ('favorited', 'category'), ('retweeted', 'datetime'), ('retweeted', 'category'), ('possibly_sensitive', 'datetime'), ('possibly_sensitive', 'category'), ('num_hashtags', 'datetime'), ('num_hashtags', 'category'), ('num_urls', 'datetime'), ('num_urls', 'category'), ('num_mentions', 'datetime'), ('num_mentions

"But no dataset can generalize well on all other datasets." - 1099
Basically, construct a random forest on one dataset, evaluate it on another dataset.

Basically, create an AUC table.
seperability - averaging across columns
generalizability - averaging across rows

They find no correlation between separability and generalizabilitty, r=0.18, p=0.6

AUC - area under ROC curve (plotting false positives as X vs true positives as Y), want close to 1.0

RandomForest 100 trees
5-fold CV

Chose a threshold, Different thresholds yield best ac-
curacy in cross-validation (F1 = 0 .94, R = 0 .93, P = 0 .94)
or cross-domain testing (F1 = 0 .77, R = 0 .68, P = 0 .88)

botometer is the baseline, Yang et al. 2019 - Yang, K.-C.; Varol, O.; Davis, C. A.; Ferrara, E.; Flammini, A.; and Menczer, F. 2019. Arming the public with artificial intelligence to counter social bots. Hum. Behav. and Emerg. Tech. 1(1):48–61.

Calculate SHAP value

[RapidAPI Botometer, see X-RapidAPI-Key after a playground post request](https://rapidapi.com/OSoMe/api/botometer-pro/playground/Botometer%20X)
https://github.com/osome-iu/botometer-python
https://onlinelibrary.wiley.com/doi/epdf/10.1002/hbe2.115


```python
>>> import botometer
>>> rapidapi_key = '2acd778cf3msh07109caf26fad9cp1d2f9ajsn334346a50893'
>>> bomx = botometer.BotometerX(rapidapi_key=rapidapi_key)
>>> bomx.get_botscores_in_batch(usernames=['@OSoMe_IU', 'botometer'])
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
  File "C:\Users\chris\AppData\Local\pypoetry\Cache\virtualenvs\sjsu-2026f-TgASqdPu-py3.12\Lib\site-packages\botometer\__init__.py", line 99, in get_botscores_in_batch
    bom_resp.raise_for_status()
  File "C:\Users\chris\AppData\Local\pypoetry\Cache\virtualenvs\sjsu-2026f-TgASqdPu-py3.12\Lib\site-packages\requests\models.py", line 1167, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 500 Server Error: Internal Server Error for url: https://botometer-pro.p.rapidapi.com/botometer-x/get_botscores_in_batch
>>> 
```

In [ ]:
def dict_subset(dic, k=10):
    preview = {key: dic[key] for key in random.choices(list(dic.keys()), k=k)}
    return preview


In [ ]:
import sys
import pprint
import string
import math
import random
from itertools import permutations

# to create the lookup n-gram table
def n_gram_probabilities(text, grams, n):
    # type: (str, str, int) -> dict[str, float]
    pdf = {}
    for gram in grams:
        pdf[(gram,)*n] = 0
    for n_gram in permutations(grams, r=n):
        pdf[n_gram] = 0

    total_grams = len(text) - n
    for c in range(0, total_grams):
        n_gram = tuple(text[c:c+n])
        pdf[n_gram] += 1

    for k, v in pdf.items():
        pdf[k] = v / total_grams
        if pdf[k] == 0:
            pdf[k] = sys.float_info.min  # smallest non-zero float possible so logarithm doesnt get domain error

    return pdf


df = pd.read_pickle(r'C:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\users.pkl')


GRAMS = string.ascii_letters + '0123456789_'  # '01234ABCDabcd'
N_GRAMS = 2
LIKELIHOOD_PDF = n_gram_probabilities(''.join(df['screen_name']), GRAMS, N_GRAMS)
pprint.pprint(dict_subset(LIKELIHOOD_PDF), indent=4)
del df


def likelihood(cell, n=2, pdf=LIKELIHOOD_PDF):
    # https://arxiv.org/pdf/1812.05932
    # Beskow and Carley 2019 - Beskow, D. M., and Carley, K. M. 2019. Its all in a name: detecting and labeling bots by their name. Comp. and Math. Org. Theory 1–12.
    # likelihood of a screen name is defined by the geometric-mean likelihood of all bigrams in it.
    # We collected over 2M unique screen names and constructed the likelihood of all 3,969 possible bigrams
    # geometric mean is \prod_{n-gram in n-grams of text} p_{n-gram}
    #   taking the log is just as effective instead of multiplying small probabilities.
    n_grams = [cell[c:c+n] for c in range(0, len(cell) - n)]
    total = len(n_grams)
    summand = sum(math.log(pdf[tuple(n_gram)]) for n_gram in n_grams)
    exp = math.exp(summand / total)
    return exp


print(f"cresci_17 likelihood: {likelihood('cresci_17') * 100:0.9f}%, high likelihood")
print(f"twibot_22 likelihood: {likelihood('twibot_22') * 100:0.9f}%, high likelihood")
random_username = ''.join(random.choices(GRAMS, k=15))
print(f"{random_username} likelihood: {likelihood(random_username) * 100:0.9f}%, low likelihood")


{   ('3', 'i'): 0.00010599957600169599,
    ('D', 'q'): 2.2250738585072014e-308,
    ('H', 'G'): 2.6499894000423998e-05,
    ('K', 'W'): 5.2999788000847996e-05,
    ('U', 'Y'): 2.2250738585072014e-308,
    ('W', 'C'): 0.00013249947000212,
    ('W', 'a'): 0.000450498198007208,
    ('n', '1'): 0.000503497986008056,
    ('t', 'Z'): 2.6499894000423998e-05,
    ('u', 'O'): 2.2250738585072014e-308}
cresci_17 likelihood: 0.153887449%, high likelihood
twibot_22 likelihood: 0.083157929%, high likelihood
TskU7oJVlxfRbLN likelihood: 0.000000000%, low likelihood


In [ ]:
import re
import datetime
from collections import OrderedDict

FEATURE_TABLE = '''
feature                         type    calculation
---                             ---     ---
statuses_count                  count
followers_count                 count
friends_count                   count
favourites_count                count
listed_count                    count
default_profile                 binary
profile_use_background_image    binary
verified                        binary

user_age                        real    hour difference between probe time and creation time
tweet_freq                      real    statuses_count / user_age
followers_growth_rate           real    followers_count / user_age
friends_growth_rate             real    friends_count / user_age
favourites_growth_rate          real    favourites_count / user_age
listed_growth_rate              real    listed_count / user_age
followers_friends_ratio         real    followers_count / friends_count
screen_name_length              count   length of screen name string
num_digits_in_screen_name       count   no. digits in screen name string
name_length                     count   length of name string
num_digits_in_name              count   no. digits in name string
description_length              count   length of description string
screen_name_likelihood          real    likelihood of the screen name
'''

UTC = datetime.timezone.utc
TWENTY_EIGHTEEN = datetime.datetime(year=2018, month=1, day=1, tzinfo=UTC)
FEATURES = OrderedDict([
    # no computation
    ('statuses_count', None),
    ('followers_count', None),
    ('friends_count', None),
    ('favourites_count', None),
    ('listed_count', None),
    ('default_profile', None),
    ('profile_use_background_image', None),
    ('verified', None),
    # computed - columnar
    ('user_age', ('created_at', TWENTY_EIGHTEEN)),
    ('tweet_freq', ('statuses_count', 'user_age')),
    ('followers_growth_rate', ('followers_count', 'user_age')),
    ('friends_growth_rate', ('friends_count', 'user_age')),
    ('favourites_growth_rate', ('favourites_count', 'user_age')),
    ('listed_growth_rate', ('listed_count', 'user_age')),
    ('followers_friends_ratio', ('followers_count', 'friends_count')),
    # computed - cellular
    ('screen_name_length', ('screen_name', len)),
    ('num_digits_in_screen_name', ('screen_name', lambda cell: len(re.findall(r'\d', cell)))),
    ('name_length', ('name', len)),
    ('num_digits_in_name', ('name', lambda cell: len(re.findall(r'\d', cell)))),
    ('description_length', ('description', len)),
    ('screen_name_likelihood', ('screen_name', likelihood)),
])


df = pd.read_pickle(r'C:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\users.pkl')
df.info()
print(df)
for feature, func in FEATURES.items():
    if func is None:
        continue
    elif isinstance(func, tuple):
        tpl = func
        print(feature, tpl)
        if isinstance(tpl[1], datetime.datetime):
            left, right = tpl
            df[feature] = (right - df[left]).dt.total_seconds() / 3600
        elif callable(tpl[1]):
            left, func = tpl[0]
            df[feature] = df[left].apply(func, axis=1)  # apply such that its done per row and cell
        else:
            left, right = tpl
            df[feature] = df[left] / df[right]
    print(df.columns)


<class 'pandas.DataFrame'>
RangeIndex: 3474 entries, 0 to 3473
Data columns (total 42 columns):
 #   Column                              Non-Null Count  Dtype              
---  ------                              --------------  -----              
 0   id                                  3474 non-null   str                
 1   name                                3473 non-null   str                
 2   screen_name                         3474 non-null   str                
 3   statuses_count                      3474 non-null   str                
 4   followers_count                     3474 non-null   str                
 5   friends_count                       3474 non-null   str                
 6   favourites_count                    3474 non-null   str                
 7   listed_count                        3474 non-null   str                
 8   url                                 1266 non-null   str                
 9   lang                                3474 non-null   

TypeError: unsupported operand type(s) for /: 'str' and 'float'

# Appendix


## Changelog
- 2026-11-16 03:36 - got the shape of what they did, implemented likelihood and a few other things, realized my pickle optimizations were useless because I need them as numbers, not strings, re-running.
- 2026-11-16 01:33 - all pickles processed w/ 184:26, saved 28.465% from 105.646GB to 75.574GB
- 2026-11-13 16:32 - initial commit
